# 12 - A hosted agent that uses Foundry IQ through a toolbox

## Required permissions

- **You:** **Foundry Owner on the parent Foundry resource**, **Search Service Contributor** + **Search Index Data Reader** (or **Search Index Data Contributor**) on Search, and `az login` in the deployment tenant. Keep your Blob knowledge source from notebook 09.
- **Facilitator:** create the shared Foundry Search connection once with `setup-toolbox-connection.ps1`. After deployment, grant **Search Index Data Reader on Search to this agent's own principal**, using `grant-agent-search-access.ps1` if you cannot assign roles yourself. Participants do not need role-assignment permission when the facilitator performs this step.
- **Service identities:** Search needs its existing Blob/model grants from notebook 09. The hosted agent's Search grant is separate from the participant's or project's access. This Blob-only lab uses agent identity, not per-user SharePoint authorization.

**Attendee alias:** keep notebook 09's `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Reuse `alice01-kb-blob`; create `alice01-toolbox-kb`, `alice01-iq-toolbox`, `alice01-iq-agent` and the local `alice01-hosted-agent-iq` folder. Platform-generated identity/role-assignment IDs remain unique IDs. `TOOLBOX_CONNECTION_NAME` is a shared read-only facilitator connection, not an attendee-created object. Aliases prevent naming collisions, not enforce access isolation.

Notebook 11 built a hosted agent with a Python function tool. Here the agent gets its knowledge from a **Foundry IQ knowledge base**, reached through a **Foundry toolbox**.

```text
you --responses.create--> hosted agent (main.py)
                              | FoundryToolbox(...)   no tool code, no Search code, no keys
                              v
                        Foundry toolbox  {prefix}-iq-toolbox      <- one MCP endpoint, versioned
                              | connection: iq-workshop-search-mcp (agentic identity)
                              v
                  Knowledge base MCP endpoint on Azure AI Search   <- tool: knowledge_base_retrieve
```

Why a toolbox? The agent code only names the toolbox. Which tools it holds, and how they authenticate, is configured in Foundry, so you can change tools without redeploying the agent.

You will:

1. Create a knowledge base over the Blob knowledge source from notebook 09.
2. Create a toolbox that exposes the knowledge base's MCP endpoint as a tool.
3. Deploy a hosted agent that uses the toolbox.
4. Let the agent's own identity read from Search.
5. Ask questions and see answers grounded in your documents.

## Prerequisites

- Notebook 09 completed with `CLEAN_UP = False`, so `{prefix}-kb` and `{prefix}-kb-blob` exist.
- The facilitator ran `scripts\setup-toolbox-connection.ps1` (the shared project connection) and gave you `TOOLBOX_CONNECTION_NAME` if it is not the default `iq-workshop-search-mcp`.
- You are signed in with `az login`.
- Notebook 11 worked for you (hosted agents are available in your project).

**Why only Blob?** The toolbox connection uses the *agent's* identity, not yours. SharePoint's per-user permissions need your identity, and Web IQ needs a per-request key, so neither fits this keyless path. Blob content is shared, so it does. Notebook 10 shows the per-user route through APIM.

In [ ]:
import os
import re
import time
import uuid
import zipfile
from pathlib import Path
from uuid import UUID

import requests
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = (
    'AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_SEARCH_RESOURCE_ID',
    'AZURE_AI_PROJECT_ENDPOINT', 'AZURE_AI_MODEL_DEPLOYMENT_NAME',
)
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Ask the facilitator; see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
search_resource_id = os.environ['AZURE_SEARCH_RESOURCE_ID']
project_endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT'].rstrip('/')
model_name = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
connection_name = os.getenv('TOOLBOX_CONNECTION_NAME', 'iq-workshop-search-mcp')
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

api_version = '2026-08-01-preview'
blob_source_name = f'{prefix}-kb-blob'
knowledge_base_name = f'{prefix}-toolbox-kb'
toolbox_name = f'{prefix}-iq-toolbox'
agent_name = f'{prefix}-iq-agent'
agent_folder = root / f'{prefix}-hosted-agent-iq'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
arm_token = get_bearer_token_provider(credential, 'https://management.azure.com/.default')
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential, allow_preview=True)
print('Knowledge base:', knowledge_base_name)
print('Toolbox:', toolbox_name, '| connection:', connection_name)
print('Agent:', agent_name)

## 1. Create the knowledge base

The Blob knowledge source from notebook 09 is reused. The knowledge base copies the chat model settings from notebook 09's knowledge base, so no extra model setup is needed. This is a control-plane call straight to Search with your own token, as in earlier notebooks.

In [ ]:
response = requests.get(
    f'{search_endpoint}/knowledgesources/{blob_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{blob_source_name} not found. Complete notebook 09 first (CLEAN_UP = False).')
response.raise_for_status()
blob_source_kind = response.json()['kind']

response = requests.get(
    f'{search_endpoint}/knowledgebases/{prefix}-kb?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{prefix}-kb not found. Complete notebook 09 first (CLEAN_UP = False).')
response.raise_for_status()
base_kb = response.json()

knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Workshop policy documents from Blob storage, for the hosted agent.',
    'retrievalInstructions': f'Use {blob_source_name} for questions about company policy, travel, onboarding or documents.',
    'answerInstructions': base_kb.get('answerInstructions'),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': blob_source_name}],
    'models': base_kb['models'],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()

knowledge_base_mcp_url = f'{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version={api_version}'
print('Knowledge base ready:', knowledge_base_name)
print('MCP endpoint:', knowledge_base_mcp_url)

## 2. Create the toolbox

Every knowledge base has an **MCP endpoint**. A toolbox tool of type `mcp` points at it:

- `server_url` is the knowledge base's MCP endpoint.
- `project_connection_id` is the shared connection the facilitator created. It decides *who* calls Search: with `agentic-identity` it is the calling agent's own identity, so there is no key and no user token.
- `require_approval='never'` lets the agent call the tool without asking.

A toolbox **version** is an immutable snapshot. The first version becomes the default automatically. The agent always uses the *default* version through one stable endpoint, so later you can publish a new version and promote it without touching the agent.

In [ ]:
from azure.ai.projects.models import MCPToolboxTool

connection = project_client.connections.get(connection_name)
print('Connection:', connection.name, connection.id)

toolbox_version = project_client.toolboxes.create_version(
    name=toolbox_name,
    description='Foundry IQ knowledge base for the workshop hosted agent.',
    tools=[
        MCPToolboxTool(
            server_label='knowledge_base',
            server_url=knowledge_base_mcp_url,
            require_approval='never',
            project_connection_id=connection.id,
        ),
    ],
)
toolbox = project_client.toolboxes.get(toolbox_name)
if toolbox.default_version != toolbox_version.version:
    toolbox = project_client.toolboxes.update(toolbox_name, default_version=toolbox_version.version)
print('Toolbox', toolbox.name, '- new version', toolbox_version.version, '- default version', toolbox.default_version)

## 3. Write the agent

Compare with notebook 11: the `@tool` function is gone. `FoundryToolbox(credential)` connects to the toolbox named in the `TOOLBOX_NAME` environment variable, discovers its tools (here `knowledge_base_retrieve`) and gives them to the agent. The agent code contains no Search URL, key or knowledge base name.

In [ ]:
main_py = """import os

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import FoundryToolbox, ResponsesHostServer
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Finds the toolbox from FOUNDRY_PROJECT_ENDPOINT (set by the platform) and TOOLBOX_NAME.
toolbox = FoundryToolbox(credential)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=credential,
)

agent = Agent(
    client=client,
    instructions=(
        "You answer questions about company policy using the knowledge base tool. "
        "If the knowledge base does not contain the answer, say you do not know. "
        "Cite the documents you used."
    ),
    tools=toolbox,
    default_options={"store": False},
)

if __name__ == "__main__":
    ResponsesHostServer(agent).run()
"""

requirements_txt = """agent-framework-foundry
agent-framework-foundry-hosting>=1.0.0b261015
"""

agent_folder.mkdir(exist_ok=True)
(agent_folder / 'main.py').write_text(main_py, encoding='utf-8')
(agent_folder / 'requirements.txt').write_text(requirements_txt, encoding='utf-8')
compile(main_py, 'main.py', 'exec')
print('Wrote', [p.name for p in agent_folder.iterdir()], 'to', agent_folder)

## 4. Deploy the agent

Same deployment as notebook 11. The only difference is the `TOOLBOX_NAME` environment variable that points the agent at your toolbox. The cell then polls until the build is active (up to 15 minutes).

In [ ]:
from azure.ai.projects.models import CodeConfiguration, HostedAgentDefinition, ProtocolVersionRecord

zip_path = agent_folder / 'agent.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(agent_folder / 'main.py', 'main.py')
    bundle.write(agent_folder / 'requirements.txt', 'requirements.txt')

definition = HostedAgentDefinition(
    cpu='0.5',
    memory='1Gi',
    protocol_versions=[ProtocolVersionRecord(protocol='responses', version='2.0.0')],
    environment_variables={'AZURE_AI_MODEL_DEPLOYMENT_NAME': model_name, 'TOOLBOX_NAME': toolbox_name},
    code_configuration=CodeConfiguration(
        runtime='python_3_13',
        entry_point=['main.py'],
        dependency_resolution='remote_build',
    ),
)

with open(zip_path, 'rb') as code_stream:
    version = project_client.agents.create_version_from_code(
        agent_name,
        definition=definition,
        code=code_stream,
        description='Hosted agent grounded in a Foundry IQ knowledge base through a toolbox',
        metadata={'workshop': 'iq', 'attendee': prefix},
    )
print('Created version', version.version)

deadline = time.time() + 15 * 60
while True:
    version = project_client.agents.get_version(agent_name, version.version)
    print('status:', version.status)
    if str(version.status).lower() == 'active':
        break
    if str(version.status).lower() == 'failed':
        raise RuntimeError(f'Build failed: {version}')
    if time.time() > deadline:
        raise TimeoutError('Still not active after 15 minutes.')
    time.sleep(15)
print('Agent is ready:', agent_name, 'version', version.version)

## 5. Let the agent read from Search

The toolbox calls Search as the **agent's identity**, which exists only now that the agent is deployed. It needs the **Search Index Data Reader** role (read-only queries) on the Search service.

The cell tries to assign the role for you. If your account is not allowed to assign roles it prints the principal ID, and the facilitator runs `scripts\grant-agent-search-access.ps1 -PrincipalId <id>`. Role changes can take a minute or two to take effect.

In [ ]:
principal_id = version.instance_identity.principal_id
print('Agent identity (principal ID):', principal_id)

search_index_data_reader = '1407120a-92aa-4202-b7e9-c0e197c71c8f'
assignment_name = uuid.uuid5(uuid.NAMESPACE_URL, f'{search_resource_id}/{principal_id}/{search_index_data_reader}')
response = requests.put(
    f'https://management.azure.com{search_resource_id}/providers/Microsoft.Authorization/roleAssignments/{assignment_name}?api-version=2022-04-01',
    headers={'Authorization': f'Bearer {arm_token()}'},
    json={'properties': {
        'roleDefinitionId': f'{search_resource_id}/providers/Microsoft.Authorization/roleDefinitions/{search_index_data_reader}',
        'principalId': principal_id,
        'principalType': 'ServicePrincipal',
    }}, timeout=(10, 60),
)
if response.status_code in (200, 201):
    print('Granted Search Index Data Reader. Wait a minute or two before asking questions.')
elif response.status_code == 409:
    print('The agent already has Search Index Data Reader.')
elif response.status_code == 403:
    print('You cannot assign roles. Ask the facilitator to run:')
    print(f'  .\\scripts\\grant-agent-search-access.ps1 -PrincipalId {principal_id}')
else:
    response.raise_for_status()

## 6. Ask the agent

The agent decides to call `knowledge_base_retrieve`, the toolbox forwards the call to your knowledge base as the agent, and the model answers from what comes back. The `output` list shows the tool call. If you see `401`/`403` or "I don't know" for a question your documents answer, the role from section 5 has probably not propagated yet; wait and ask again.

In [ ]:
openai_client = project_client.get_openai_client(agent_name=agent_name)

response = openai_client.responses.create(input='What is our policy for booking international travel?')
print(response.output_text)
print()
for item in response.output:
    print('output item:', item.type, getattr(item, 'name', '') or '')

## 7. Prove it is grounded

A question the documents cannot answer should get "I don't know" instead of a guess.

In [ ]:
response = openai_client.responses.create(input='What is the capital of Australia?')
print(response.output_text)

## Cleanup

Set `CLEAN_UP = True` to delete your agent, toolbox and knowledge base. Notebook 09's sources and knowledge base, and the shared connection, are not touched.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    project_client.agents.delete(agent_name)
    project_client.toolboxes.delete(toolbox_name)
    response = requests.delete(
        f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 60),
    )
    if response.status_code != 404:
        response.raise_for_status()
    print('Deleted', agent_name, toolbox_name, knowledge_base_name)
else:
    print('Cleanup skipped. Set CLEAN_UP = True to delete', agent_name, toolbox_name, knowledge_base_name)